# Spearman correlations between pXRF elements and soil variables

Reproduces Figure A.1. Only correlations with absolute Spearman's rho of at least 0.40 are annotated, as in the final appendix.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
# ============================================================
# FIGURE A.1 — pXRF CORRELATION HEATMAP
# ------------------------------------------------------------
# Objective:
#   Generate a professional heatmap of Spearman correlations
#   between selected pXRF-derived elements and soil/biological variables.
#
# Output:
#   Figure S1: pXRF correlations with soil physicochemical
#              and biological variables.
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------------------
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

soil_file = PROJECT_DIR / "data" / "soil_data.xlsx"
pxrf_file = PROJECT_DIR / "data" / "pxrf_data.csv"
sheet_name = "PredictionNewSamplesSalinity_Fi"

out_dir = PROJECT_DIR / "outputs" / "04_pxrf_correlations"
out_dir.mkdir(parents=True, exist_ok=True)

for required_file in [soil_file, pxrf_file]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {required_file}\n"
            "Place the soil and pXRF files in the data folder or update the paths."
        )

# ------------------------------------------------------------
# 2. LOAD DATA
# ------------------------------------------------------------
soil_df = pd.read_excel(soil_file, sheet_name=sheet_name)
pxrf_df = pd.read_csv(pxrf_file)

soil_df.columns = soil_df.columns.str.strip()
pxrf_df.columns = pxrf_df.columns.str.strip()

soil_df["SSN"] = soil_df["SSN"].astype(str).str.strip()
pxrf_df["SSN"] = pxrf_df["SSN"].astype(str).str.strip()

# ------------------------------------------------------------
# 3. CLEAN SOIL DATA
# ------------------------------------------------------------
soil_df["Zone"] = soil_df["SITE"].astype(str).str.strip().str.upper()
soil_df = soil_df[soil_df["Zone"].isin(["PERKERRA", "TANA"])].copy()

soil_rename = {
    "EC 1:2 (dS/cm)": "EC_spectral",
    "EC (dS/m) 1:1": "EC_lab",
    "ESP est": "ESP",
    "SOC (%)": "SOC",
    "Clay (%)": "Clay",
    "CEC cmolc/kg": "CEC",
    "TN (mg/kg)": "TN",
    "TotalBacteria": "TotalBacteria",
    "TotalFungi": "TotalFungi",
    "TYPES_BACTERIA": "BacterialRichness",
    "TYPES_FUNGI": "FungalRichness"
}

soil_df = soil_df.rename(columns=soil_rename)

target_vars = [
    "EC_spectral",
    "EC_lab",
    "ESP",
    "pH",
    "SOC",
    "TN",
    "Clay",
    "CEC",
    "TotalBacteria",
    "TotalFungi",
    "BacterialRichness",
    "FungalRichness"
]

target_vars = [v for v in target_vars if v in soil_df.columns]

for col in target_vars:
    soil_df[col] = pd.to_numeric(soil_df[col], errors="coerce")

# ------------------------------------------------------------
# 4. CLEAN pXRF ELEMENTS
# ------------------------------------------------------------
element_cols_raw = [c for c in pxrf_df.columns if "(mg/kg)" in c]

for col in element_cols_raw:
    pxrf_df[col] = (
        pxrf_df[col]
        .astype(str)
        .str.replace("<LOD", "", regex=False)
        .str.strip()
    )
    pxrf_df[col] = pd.to_numeric(pxrf_df[col], errors="coerce")

pxrf_rename = {}

for col in element_cols_raw:
    clean = (
        col.replace(" (mg/kg)", "")
           .replace(" ", "_")
           .replace("/", "_")
           .replace("-", "_")
    )
    pxrf_rename[col] = f"pXRF_{clean}"

pxrf_df = pxrf_df.rename(columns=pxrf_rename)
element_cols = list(pxrf_rename.values())

# ------------------------------------------------------------
# 5. MERGE BY SSN
# ------------------------------------------------------------
merged = soil_df[["SSN", "Zone"] + target_vars].merge(
    pxrf_df[["SSN"] + element_cols],
    on="SSN",
    how="left",
    validate="one_to_one"
)

print("Merged dataset:", merged.shape)
print("Samples with pXRF data:", merged[element_cols].notna().any(axis=1).sum())

# ------------------------------------------------------------
# 6. SELECT FINAL pXRF ELEMENTS
# ------------------------------------------------------------
# Elements selected for the final heatmap.
# These are interpretable and avoid overcrowding the figure.

selected_elements = [
    "pXRF_Al",
    "pXRF_Ca",
    "pXRF_Fe",
    "pXRF_K",
    "pXRF_Mg",
    "pXRF_Mn",
    "pXRF_P",
    "pXRF_Rb",
    "pXRF_S",
    "pXRF_Sr",
    "pXRF_Th",
    "pXRF_Ti",
    "pXRF_U",
    "pXRF_Zr"
]

selected_elements = [e for e in selected_elements if e in merged.columns]

# Optional: remove elements with too many missing values
missing_pct = merged[selected_elements].isna().mean() * 100
selected_elements = missing_pct[missing_pct <= 30].index.tolist()

print("\nElements used in heatmap:")
print(selected_elements)

# ------------------------------------------------------------
# 7. SPEARMAN CORRELATIONS
# ------------------------------------------------------------
corr_rows = []

for elem in selected_elements:
    for var in target_vars:
        pair = merged[[elem, var]].dropna()

        if len(pair) >= 5:
            rho, pval = spearmanr(pair[elem], pair[var])
        else:
            rho, pval = np.nan, np.nan

        corr_rows.append({
            "pXRF_element": elem,
            "Target_variable": var,
            "N": len(pair),
            "Spearman_rho": rho,
            "p_value": pval
        })

corr_df = pd.DataFrame(corr_rows)

corr_df.to_excel(
    os.path.join(out_dir, "figure_A_1_pXRF_spearman_correlations.xlsx"),
    index=False
)

# ------------------------------------------------------------
# 8. PROFESSIONAL LABELS
# ------------------------------------------------------------
element_labels = {
    "pXRF_Al": "Al",
    "pXRF_Ca": "Ca",
    "pXRF_Fe": "Fe",
    "pXRF_K": "K",
    "pXRF_Mg": "Mg",
    "pXRF_Mn": "Mn",
    "pXRF_P": "P",
    "pXRF_Rb": "Rb",
    "pXRF_S": "S",
    "pXRF_Sr": "Sr",
    "pXRF_Th": "Th",
    "pXRF_Ti": "Ti",
    "pXRF_U": "U",
    "pXRF_Zr": "Zr"
}

target_labels = {
    "EC_spectral": r"EC$_{1:2}$ spectral",
    "EC_lab": r"EC$_{1:1}$ lab",
    "ESP": "ESP",
    "pH": "pH",
    "SOC": "SOC",
    "TN": "TN",
    "Clay": "Clay",
    "CEC": "CEC",
    "TotalBacteria": "Total bacteria",
    "TotalFungi": "Total fungi",
    "BacterialRichness": "Bacterial richness",
    "FungalRichness": "Fungal richness"
}

# Recommended order for interpretation
target_order = [
    "EC_spectral",
    "EC_lab",
    "ESP",
    "pH",
    "SOC",
    "TN",
    "Clay",
    "CEC",
    "TotalBacteria",
    "TotalFungi",
    "BacterialRichness",
    "FungalRichness"
]

target_order = [v for v in target_order if v in target_vars]

element_order = [
    "pXRF_Al",
    "pXRF_Ca",
    "pXRF_Fe",
    "pXRF_K",
    "pXRF_Mg",
    "pXRF_Mn",
    "pXRF_P",
    "pXRF_Rb",
    "pXRF_S",
    "pXRF_Sr",
    "pXRF_Th",
    "pXRF_Ti",
    "pXRF_U",
    "pXRF_Zr"
]

element_order = [e for e in element_order if e in selected_elements]

corr_matrix = corr_df.pivot(
    index="pXRF_element",
    columns="Target_variable",
    values="Spearman_rho"
)

corr_matrix = corr_matrix.loc[element_order, target_order]

corr_matrix = corr_matrix.rename(
    index=element_labels,
    columns=target_labels
)

# ------------------------------------------------------------
# 9. ANNOTATION MATRIX
# ------------------------------------------------------------
# Annotate only moderate-to-strong correlations.
# This keeps the figure clean.

annotation_threshold = 0.40

annot_matrix = corr_matrix.copy().astype(object)

for i in range(corr_matrix.shape[0]):
    for j in range(corr_matrix.shape[1]):
        val = corr_matrix.iloc[i, j]

        if pd.notna(val) and abs(val) >= annotation_threshold:
            annot_matrix.iloc[i, j] = f"{val:.2f}"
        else:
            annot_matrix.iloc[i, j] = ""

# ------------------------------------------------------------
# 10. FINAL HEATMAP
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

fig_width = 15.5
fig_height = max(7.5, 0.48 * len(corr_matrix.index))

fig, ax = plt.subplots(figsize=(fig_width, fig_height))

heat = sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.45,
    linecolor="white",
    annot=annot_matrix,
    fmt="",
    annot_kws={
        "size": 11,
        "weight": "bold"
    },
    cbar_kws={
        "label": "Spearman's ρ",
        "shrink": 0.92,
        "pad": 0.03
    },
    ax=ax
)

ax.set_xlabel("")
ax.set_ylabel("pXRF-derived elements", fontsize=13)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=40,
    ha="right",
    rotation_mode="anchor"
)

ax.set_yticklabels(
    ax.get_yticklabels(),
    rotation=0
)

# Improve colorbar label size
cbar = heat.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)
cbar.set_label("Spearman's ρ", fontsize=13)

# Make annotation color readable depending on cell value
for text in ax.texts:
    try:
        value = float(text.get_text())
        if abs(value) >= 0.60:
            text.set_color("white")
        else:
            text.set_color("black")
    except ValueError:
        pass

plt.tight_layout()

png_path = os.path.join(out_dir, "figure_A_1_pXRF_correlation_heatmap_professional.png")
tiff_path = os.path.join(out_dir, "figure_A_1_pXRF_correlation_heatmap_professional.tiff")
pdf_path = os.path.join(out_dir, "figure_A_1_pXRF_correlation_heatmap_professional.pdf")

plt.savefig(png_path, dpi=600, bbox_inches="tight")
plt.savefig(tiff_path, dpi=600, bbox_inches="tight")
plt.savefig(pdf_path, bbox_inches="tight")

plt.show()

print("\nFigure S1 saved as:")
print(png_path)
print(tiff_path)
print(pdf_path)
